## Define

In [15]:
from resources.utils import *
from gensim.models import Word2Vec

!mkdir -p tb
!mkdir -p tb/embeddings

mc['github'].list_collection_names()

['users_followers_embeddings',
 'followers',
 'projects',
 'users_following_embeddings',
 'followers_all',
 'users_watches',
 'repos_watches_embeddings',
 'repos_stars_embeddings',
 'users',
 'following']

## Repos

In [16]:
repo_model_name = 'repos_watches_emb_sg_d300_w50_mc200_e50'
repo_model = Word2Vec.load('./models/' + repo_model_name + '.bin')
print(repo_model)

/opt/conda/lib/python3.7/site-packages/smart_open/smart_open_lib.py:398: UserWarning:

This function is deprecated, use smart_open.open instead. See the migration notes for details: https://github.com/RaRe-Technologies/smart_open/blob/master/README.rst#migrating-to-the-new-open-function



Word2Vec(vocab=84973, size=300, alpha=0.025)


In [17]:
good_rids = []

for rid, r in repo_model.wv.vocab.items():
    if r.count >= 1_000:
        good_rids.append(int(rid))
len(good_rids)

18722

In [18]:
good_rids

[19,
 1226,
 280,
 11603,
 284,
 2561674,
 3189,
 5831963,
 101520307,
 348,
 6180,
 288,
 10313229,
 2812,
 231,
 3981,
 18089783,
 847,
 3085,
 853,
 992,
 14335,
 1362,
 37811,
 1036,
 29261651,
 1344863,
 12031,
 5874,
 337,
 10735,
 1267,
 183,
 1907512,
 964,
 9427,
 1301,
 1691,
 6801328,
 1707,
 1717,
 5101,
 1566,
 1312,
 3601879,
 1911,
 9316,
 7074,
 19034,
 5461,
 2007,
 202,
 17530,
 50703945,
 60488339,
 1229,
 2067,
 4564,
 112576,
 3430,
 2157,
 2160,
 2184,
 9673869,
 11879,
 2233,
 10547585,
 10642813,
 4488,
 2310,
 1155356,
 11313132,
 19626,
 2326,
 4477,
 392,
 2602,
 581,
 5049,
 6313,
 4455,
 31810056,
 8109,
 2279,
 7608,
 3054,
 1334,
 8029,
 11833707,
 3231,
 1896,
 103284,
 89100763,
 93613877,
 3470,
 588,
 553,
 20783,
 33986963,
 25725,
 5711,
 1713,
 3777,
 6093,
 2183,
 10559,
 217063,
 4041,
 9862004,
 4067,
 6782,
 30141,
 4136,
 15762,
 295,
 114167,
 852,
 4492,
 20629,
 61168739,
 72272853,
 1088,
 13454178,
 4612,
 35704,
 129770,
 4788,
 1298,
 5

In [19]:
def dropCategories(df, col, threshold):
    drop_list = df[col].value_counts()[threshold:].index.to_list()
    df[col].cat.remove_categories(drop_list, inplace=True)
    return df

In [20]:
repos_cur = mc['github']['projects'].find({"_id": {"$in": good_rids}}, ['url', 'language', 'created_at', 'updated_at'])
repos_emb_cur = mc['github']['repos_stars_embeddings'].find({"_id": {"$in": good_rids}})

In [21]:
repos_df = pd.DataFrame(list(repos_cur)).set_index('_id')[['url', 'language', 'created_at', 'updated_at']]
repos_df.language = repos_df.language.astype("category")
repos_df = dropCategories(repos_df, "language", 42)
repos_df

,url,language,created_at,updated_at
_id,,,,
6,cocos2d/cocos2d-x,C++,2010-11-18 23:17:00,2019-02-26 20:51:59
19,samuelclay/NewsBlur,Objective-C,2009-01-05 14:00:43,2019-02-26 03:37:41
29,opencv/opencv,C++,2012-07-19 09:40:17,2016-10-11 14:11:27
34,pockethub/PocketHub,JavaScript,2011-09-08 16:52:50,2016-10-11 14:11:27
37,angular/angular.js,JavaScript,2010-01-06 00:34:37,2019-02-27 03:14:26
49,alexgorbatchev/syntaxhighlighter,NaN,2014-10-07 13:48:58,2019-01-07 05:02:27
55,yui/yui3,JavaScript,2008-12-05 19:12:24,2019-02-26 04:45:11
71,llvm-mirror/llvm,NaN,2012-01-27 23:49:56,2019-02-26 12:59:44
74,joyent/libuv,C,2011-03-29 21:10:45,2019-02-14 03:06:31


In [22]:
repos_emb_df = pd.DataFrame(list(repos_emb_cur)).set_index('_id')
repos_emb_df

,n_vecs,vec
_id,,
6,13901,"[0.24193669855594635, 0.1203954815864563, 0.49..."
19,5455,"[0.1151142343878746, 0.117125004529953, 0.4360..."
29,7744,"[0.17094147205352783, -0.06545144319534302, 0...."
34,5588,"[0.22444790601730347, -0.04315068945288658, 0...."
37,56025,"[0.23754794895648956, -0.08706655353307724, 0...."
49,3550,"[0.18847320973873138, -0.02633054554462433, 0...."
55,3783,"[0.28016871213912964, -0.024500740692019463, 0..."
71,3743,"[0.32852479815483093, -0.20795677602291107, 0...."
74,2289,"[0.38658544421195984, -0.2022801786661148, 0.4..."


In [25]:
repos_emb_df.columns = ['n_vecs', 'embeddings']
repos_df = repos_df.join(repos_emb_df)
repos_df

ValueError: columns overlap but no suffix specified: Index(['n_vecs', 'embeddings'], dtype='object')

In [26]:
repos_df.language.value_counts()

JavaScript          3920
Java                1700
Python              1607
Go                   843
Objective-C          718
C++                  691
PHP                  667
C                    630
Ruby                 585
HTML                 462
Swift                453
C#                   362
CSS                  349
Shell                338
TypeScript           196
Rust                 110
VimL                 108
CoffeeScript         101
Scala                 99
Jupyter Notebook      87
Clojure               59
Lua                   47
Vue                   44
Kotlin                42
Haskell               39
Emacs Lisp            32
Perl                  29
Makefile              26
TeX                   26
R                     25
PowerShell            23
Elixir                22
OCaml                 21
Groovy                19
Erlang                19
Assembly              17
Dart                  15
Objective-C++         12
Matlab                11
Common Lisp            7


In [28]:
repos_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 18670 entries, 6 to 127439965
Data columns (total 6 columns):
url           18670 non-null object
language      14573 non-null category
created_at    18670 non-null datetime64[ns]
updated_at    18670 non-null datetime64[ns]
n_vecs        18670 non-null int64
embeddings    18670 non-null object
dtypes: category(1), datetime64[ns](2), int64(1), object(2)
memory usage: 1.5+ MB


In [25]:
repos_df.to_pickle("./tb/embeddings/repos_1k_gte10k.pkl")

In [14]:
repos_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 970 entries, 6 to 119888241
Data columns (total 4 columns):
url           970 non-null object
language      764 non-null category
created_at    970 non-null datetime64[ns]
updated_at    970 non-null datetime64[ns]
dtypes: category(1), datetime64[ns](2), object(1)
memory usage: 32.8+ KB


## Followers

In [199]:
followers_model_name = 'users_followers_emb_sg_d300_w30_mc3_e5'
followers_model = Word2Vec.load('./models/' + followers_model_name + '.bin')
print(followers_model)

Word2Vec(vocab=1446832, size=300, alpha=0.025)


In [222]:
followers_good_uids = []

for uid, r in model.wv.vocab.items():
    if r.count >= 100:
        followers_good_uids.append(int(uid))
len(followers_good_uids)

24435

In [223]:
followers_users_cur = mc['github']['users'].find({"_id": {"$in": followers_good_uids}}, 
                                       ['login', 'type', 'fake', 'location', 'deleted', 'country_code', 'created_at'])
followers_users_emb_cur = mc['github']['users_followers_embeddings'].find({"_id": {"$in": followers_good_uids}})

followers_users_df = pd.DataFrame(list(followers_users_cur)).set_index('_id')[
    ['login', 'type', 'fake', 'location', 'deleted', 'country_code', 'created_at']
]
followers_users_emb_df = pd.DataFrame(list(followers_users_emb_cur)).set_index('_id')
followers_users_emb_df.columns = ['n_vecs', 'embeddings']
followers_users_df = followers_users_df.join(followers_users_emb_df)
followers_users_df

,login,type,fake,location,deleted,country_code,created_at,n_vecs,embeddings
_id,,,,,,,,,
2,jmettraux,USR,False,Hiroshima,False,jp,2008-03-22 00:37:42,430,"[0.2063964605331421, 0.3219830095767975, 0.054..."
4,kennethkalmer,USR,False,"Johannesburg, South Africa",False,za,2008-04-28 17:25:53,102,"[0.2238793522119522, 0.2812013626098633, 0.088..."
7,hasimo,USR,False,Tokyo/Japan,False,jp,2008-09-30 00:05:24,256,"[0.3436596989631653, 0.6199936866760254, -0.04..."
10,mcollina,USR,False,In the clouds above Italy,False,None,2009-02-05 21:24:19,371,"[0.32903146743774414, 0.4905785024166107, 0.22..."
47,l4u,USR,False,Hong Kong,False,hk,2009-06-22 14:13:10,309,"[0.3731025159358978, 0.5073902010917664, 0.017..."
49,corefan,USR,False,None,False,None,2012-03-18 01:43:30,386,"[0.33945879340171814, -0.07949918508529663, 0...."
58,hcilab,ORG,False,"Fredericton, New Brunswick, Canada",False,ca,2012-04-04 03:12:28,21738,"[0.38965266942977905, 0.8585405349731445, 0.42..."
70,rafacm,USR,False,Vienna,False,at,2010-07-22 03:58:01,129,"[0.28958895802497864, 0.16287967562675476, 0.1..."
109,jfirebaugh,USR,False,"San Francisco, CA",False,us,2009-06-24 18:01:39,127,"[0.18331538140773773, 0.30185845494270325, 0.1..."


In [224]:
followers_users_df.to_pickle("./tb/embeddings/users_followers_24k_gte100.pkl")

## Following

In [225]:
following_model_name = 'users_following_emb_sg_d300_w30_mc3_e5'
following_model = Word2Vec.load('./models/' + following_model_name + '.bin')
print(following_model)

Word2Vec(vocab=1385758, size=300, alpha=0.025)


In [244]:
following_good_uids = []

for uid, r in following_model.wv.vocab.items():
    if r.count >= 1_000:
        following_good_uids.append(int(uid))
len(following_good_uids)

1537

In [245]:
following_users_cur = mc['github']['users'].find({"_id": {"$in": following_good_uids}}, 
                                       ['login', 'type', 'fake', 'location', 'deleted', 'country_code', 'created_at'])
following_users_emb_cur = mc['github']['users_following_embeddings'].find({"_id": {"$in": following_good_uids}})

following_users_df = pd.DataFrame(list(following_users_cur)).set_index('_id')[
    ['login', 'type', 'fake', 'location', 'deleted', 'country_code', 'created_at']
]
following_users_emb_df = pd.DataFrame(list(following_users_emb_cur)).set_index('_id')
following_users_emb_df.columns = ['n_vecs', 'embeddings']
following_users_df = following_users_df.join(following_users_emb_df)
following_users_df

,login,type,fake,location,deleted,country_code,created_at,n_vecs,embeddings
_id,,,,,,,,,
10,mcollina,USR,False,In the clouds above Italy,False,None,2009-02-05 21:24:19,2574,"[-0.13591870665550232, 0.30812159180641174, -0..."
27,jimweirich,USR,False,Cincinnati,False,us,2008-03-27 18:19:22,1905,"[-0.5672861933708191, 0.192423477768898, -0.17..."
78,samuelclay,USR,False,"Cambridge, Massachusetts",False,us,2009-01-05 03:46:08,1295,"[-0.44099417328834534, 0.1900487095117569, -0...."
129,tenderlove,USR,False,Seattle,False,us,2008-03-14 19:04:17,7904,"[-0.43326085805892944, 0.38780757784843445, -0..."
189,hmason,USR,False,"New York, NY",False,us,2008-07-22 15:55:09,3413,"[-0.27972522377967834, 0.024197496473789215, -..."
216,fredwu,USR,False,"Melbourne, Australia (& Shanghai, China)",False,None,2008-10-31 03:34:48,1244,"[-0.7697437405586243, -0.10295889526605606, -0..."
272,yui,ORG,False,"Sunnyvale, CA",False,us,2008-12-04 00:46:11,1270,"[-0.8946189284324646, -0.1585889607667923, -0...."
303,Marak,USR,False,"New York City, New York",False,us,2009-04-03 01:34:57,2436,"[-0.42212310433387756, 0.15440842509269714, -0..."
376,tianyicui,USR,False,New York City,False,us,2009-02-09 13:50:20,1614,"[-0.5902652144432068, 0.044978946447372437, -0..."


In [246]:
following_users_df.to_pickle("./tb/embeddings/users_following_2k_gte1k.pkl")

## sandbox

In [1]:
fls = !ls ./tb/embeddings/

for f in fls:
    tmp_df = pd.read_pickle(f'./tb/embeddings/{f}')
    print(f'{f} - {len(tmp_df)} - gte{tmp_df["n_vecs"].min()}')

repos_19k_gte1k.pkl - 18670 - gte1000
repos_1k_gte10k.pkl - 970 - gte10002
repos_85k_gte200.pkl - 84806 - gte200
users_followers_24k_gte100.pkl - 24435 - gte100
users_followers_70k_gte50.pkl - 69570 - gte50
users_following_29k_gte100.pkl - 28827 - gte100
users_following_2k_gte1k.pkl - 1537 - gte1000
users_following_88k_gte40.pkl - 88323 - gte40
